# Round 3 Gloves Off — 市场初探

本轮有 12 个产品：1 个 underlying (`VELVETFRUIT_EXTRACT`) + 10 个 voucher (`VEV_4000~6500`) + 1 个无关的 `HYDROGEL_PACK`。

这个 notebook 的目标是**先把市场看懂**，所以：
- 前半部分完全不涉及期权理论，只看订单簿、成交、点差。
- 期权相关的抽象图（smile、strike arb）放到后半部分，在你对数据有感觉后再看。

---
## 期权速通（3 分钟版）

把 `VEV_5000` 想成一张**彩票**：你现在花点钱买，换来一个权利：**到期时**，你可以（但不一定要）用 5000 的价格去买 1 份 `VELVETFRUIT_EXTRACT`。

- 如果到期时 VELVETFRUIT 涨到 5300，这张彩票就值 **300**（你用 5000 买到 5300 的东西，净赚 300）。
- 如果到期时 VELVETFRUIT 只有 4900，彩票就是**废纸**，价值 0（没人愿意用 5000 去买 4900 的东西）。
- 所以彩票的最终价值 = `max(0, VELVETFRUIT - strike)`。永远 ≥ 0。

10 张彩票的区别只是 strike（行权价）：
- `VEV_4000` —— strike 很低，几乎肯定会行权。价格几乎就是 `VELVETFRUIT - 4000`，跟着 underlying 1:1 涨跌。
- `VEV_5100~5500` —— strike 在 underlying 附近，**最敏感**：underlying 涨一点，这张彩票涨很多（百分比意义上）。交易最活跃。
- `VEV_6500` —— strike 很高，除非 underlying 暴涨，否则废纸一张。价格接近 0。

### 盘口是什么

每个产品都有一个**订单簿**（order book）：
- **bid** = 买家挂的单（想以这个价格买）；bid_price_1 是最好（最高）的买价。
- **ask** = 卖家挂的单（想以这个价格卖）；ask_price_1 是最好（最低）的卖价。
- **spread** = ask_1 - bid_1，你每做一笔 round-trip 至少要付的摩擦成本。
- 数据里给了 3 档（L1/L2/L3），反映深度（有多少量愿意在各个价位成交）。

### 本 notebook 的时间轴

3 天历史数据合在一起，x 轴用 `global_ts = day × 1,000,000 + timestamp`：
- day 0: 0 ~ 1M （期权还有 8 天到期）
- day 1: 1M ~ 2M （还有 7 天）
- day 2: 2M ~ 3M （还有 6 天）

竖直虚线就是 day 分界。

In [ ]:
from vev_plot import Context, enable_plotly_resample
from vev_plot.plots import (
    plot_underlying_detail, plot_product_detail, plot_depth_grid,
    plot_overview, plot_spread, plot_activity, plot_day_overlay,
    plot_moneyness, plot_iv_smile, plot_iv_residual, plot_price_residual,
    plot_underlying_autocorr, plot_iv_surface_overlay, plot_strike_arb,
)

enable_plotly_resample(max_points=5000)
ctx = Context.from_data_dir("data")
print(f"days={ctx.days}  products={len(ctx.products)}  ob_rows={ctx.ob_wide.height:,}  trades={ctx.trades.height:,}")

## 0. HYDROGEL_PACK —— 无关产品的微观结构

先看这张"与期权无关"的产品的盘口微观结构。全部图都用 `price − wall_mid` 归一化，y=0 虚线即 fair price（wall_mid）。紫点 = 主动买（成交价 > wall_mid），黄点 = 主动卖（成交价 < wall_mid）。

In [ ]:
plot_product_detail(ctx, 'HYDROGEL_PACK', normalize=True).show()
plot_product_detail(ctx, 'HYDROGEL_PACK', normalize=False).show()

## 1. Underlying —— 先看最简单的产品

`VELVETFRUIT_EXTRACT` 就是个普通资产，没有期权那些复杂性。先熟悉一下图里的元素：

- **绿点** = 买单（bid）；深绿=L1（最好价），浅绿=L2/L3
- **红点** = 卖单（ask）；深红=L1，浅红=L2/L3
- **点大小** ∝ 这档挂的量
- **黑 X** = 真实成交（trades_*.csv 里的一笔笔 fill）

竖虚线是 day 分界。hover 任何点可以看 price/volume/level/day/ts。

In [ ]:
# trade_volume_filter: 仅显示 quantity 落在该集合中的成交点；None 显示全部成交
trade_volume_filter = {8}  # e.g. {1, 5, 10}

plot_underlying_detail(ctx, normalize=True, trade_volume_filter=trade_volume_filter).show()
plot_underlying_detail(ctx, normalize=False, trade_volume_filter=trade_volume_filter).show()

## 2. 一个 voucher 长什么样 —— VEV_5200（接近 ATM）

挑 `VEV_5200`，因为 underlying 在 5200 附近晃，这张 voucher 是最敏感的一档。

对比上一张图你会看到：
- **价格量级差一个数量级**（voucher ~80, underlying ~5200）——因为 voucher 只是个权利，不是资产本身。
- **成交相对稀疏**——voucher 流动性比 underlying 差。
- **点差占比更大**——同样一单的摩擦成本相对 voucher 价格是很贵的。

换一个 strike（比如 `plot_product_detail(ctx, 'VEV_4000')` vs `'VEV_6500'`）可以看到不同深度的 voucher 长什么样。

In [ ]:
plot_product_detail(ctx, 'VEV_5100', normalize=True).show()
plot_product_detail(ctx, 'VEV_5100', normalize=False).show()

## 3. 10 个 voucher 批量对比

5×2 格子，每格一个 strike。注意三件事：

1. **价格量级随 strike 变化**：左上（VEV_4000）价格在 1200 以上，右下（VEV_6500）几乎是 0——strike 越低的 voucher 越像 underlying 本身。
2. **流动性不均匀**：哪些 strike 有持续成交（黑 X 多）、哪些几乎没人交易——这决定了你将来能不能做 MM。
3. **点差差异**：深 OTM（6000/6500）的 spread 相对价格经常是 100%+；近 ATM（5100~5500）相对更紧。

参数 `only_level_1=True` 只画 L1 最好价，视觉更干净。

In [ ]:
plot_depth_grid(ctx, normalize=True).show()
plot_depth_grid(ctx, normalize=False).show()

## 4. 大局视角 —— 所有产品 mid 叠加

散点图太密的时候，用线图看大趋势。左轴 underlying（黑），右轴 10 条 voucher（按 strike 渐变色）。hover 会一条竖线读出所有产品在同一时刻的值。

In [ ]:
plot_overview(ctx).show()

## 5. Spread 时序 + 分布

每个 voucher 的 ask_1 - bid_1 随时间的变化，以及分布直方图。长尾告诉你最坏情形的摩擦成本——MM 做多深交易受点差主导。

In [ ]:
plot_spread(ctx).show()

## 6. 成交活跃度 heatmap

每列是一段时间（默认 20k ts），每行是一个产品。颜色越亮表示这段时间内这产品的成交笔数越多。决定了你往哪几个产品上倾斜精力。

In [ ]:
plot_activity(ctx, time_bin=20_000).show()

## 7. Day overlay —— 日内形态是否重复

把三天叠到同一 day-local 时间轴（`normalize=True` 时都从 0 开始）。如果三条轨迹高度重合，就说明存在稳定的日内 pattern 可以利用。

In [ ]:
plot_day_overlay(ctx, products=['VELVETFRUIT_EXTRACT', 'VEV_5000', 'VEV_5200', 'VEV_5400'], normalize=True).show()

---
## 下面是期权专有视角（看完前 7 节再看）

### 8. Moneyness 视图

把 voucher 的相对贵贱画在一张图上。x 轴是 **moneyness** `m_t = ln(K/S_t) / sqrt(T)`，这个量把不同 strike、不同 TTE 的期权拉到可比的尺度。

- `m_t < 0` → ITM（深度价内，strike 低于 underlying）
- `m_t = 0` → ATM
- `m_t > 0` → OTM（深度价外）

如果三天的点阵形状类似，说明 pricing 结构稳定（可以 fit smile）；如果形状变了，说明市场定价逻辑在漂移。

In [ ]:
plot_moneyness(ctx).show()

### 9. 无套利检查

相邻 strike 的 call 价差必须落在 `[0, strike差]` 之间——这是纯数学约束，跟建模无关。任何穿顶/破底都是无风险套利机会（受点差和仓位限制实际化）。

In [ ]:
plot_strike_arb(ctx).show()

---
## 下一步

1. `vev_plot/fair/<PRODUCT>.py` —— 给每个产品定义 fair value（先用 wall_mid 占位即可）。
2. `vev_plot/options.py` —— Black-Scholes + implied_vol 求解。
3. 基于 1+2，新增 `plot_iv_timeseries` / `plot_iv_smile` / `plot_iv_residual`——这才开始进入 IV scalping / vol arbitrage 的范畴。

### 10. Implied Vol Surface Overlay

把每个 voucher 的 mid 价格反解成 IV（Black-Scholes, r=0），并将各 strike 的 IV 曲面叠加到同一个 3D Plotly 中。

In [ ]:
plot_iv_surface_overlay(ctx, sample_every=40).show()

### 11-14. 方法论补充绘图

按文档一次性补充：IV Smile、IV Residual、Price Residual、Underlying Autocorr vs MC baseline。

In [ ]:
plot_iv_smile(ctx, sample_every=10).show()
plot_iv_residual(ctx, sample_every=10).show()
plot_price_residual(ctx, sample_every=10).show()
plot_underlying_autocorr(ctx, max_lag=100, n_sims=300).show()

In [ ]:
from vev_plot.plots.iv_surface import _iv_samples
import pandas as pd
import plotly.express as px

# 复用现有 IV 反解口径，构造所有 voucher 的 iv 时间序列
iv_df = _iv_samples(
    ctx,
    include=None,
    sample_every=10,
    rate=0.0,
    sigma_lo=1e-4,
    sigma_hi=3.0,
    tol=1e-6,
    max_iter=100,
    price_eps=1e-6,
)

if iv_df.is_empty():
    raise ValueError("No valid IV samples found for correlation heatmap.")

iv_pdf = iv_df.select(["global_ts", "product", "iv"]).to_pandas()
iv_wide = (
    iv_pdf.pivot_table(index="global_ts", columns="product", values="iv", aggfunc="mean")
    .sort_index()
)

# 删掉 VEV_6000 / VEV_6500，仅保留 voucher IV
drop_cols = {"VEV_6000", "VEV_6500"}
keep_cols = [c for c in iv_wide.columns if c not in drop_cols]
panel = iv_wide[keep_cols]

# 1) 原始序列相关系数热力图（仅 voucher IV）
iv_corr = panel.corr(min_periods=20)
fig = px.imshow(
    iv_corr,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    aspect="auto",
    title="Correlation Heatmap (Voucher IV ex-6000/6500)",
)
fig.update_layout(height=780, xaxis_title="Series", yaxis_title="Series")
fig.show()

# 2) 一阶差分序列相关系数热力图
iv_diff = panel.diff().dropna(how="all")
if iv_diff.empty:
    raise ValueError("No valid differenced samples found for heatmap.")

iv_diff_corr = iv_diff.corr(min_periods=20)
fig_diff = px.imshow(
    iv_diff_corr,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    aspect="auto",
    title="Correlation Heatmap of First Difference (Voucher IV ex-6000/6500)",
)
fig_diff.update_layout(height=780, xaxis_title="Series", yaxis_title="Series")
fig_diff.show()